This script uses a "cold start" of gonality savings graphs. The goal is to generate graphs (instead of timing).

**Goal:** Find Gonality Savings Graphs (V=7, max edge multiplicity 2), then iteratively refined via a reward-guided generate → score → retrain loop.

Encoding: lower-triangle, row by row. First token = vertex count + 10. EOS = 4, PAD = 5.

Requires to upload a .zip file of known_gsg's!

In [7]:
import numpy as np
import torch
import torch.nn.functional as F
import networkx as nx
import zipfile
import os
import glob

NUM_VERTICES = 9

VERTEX_OFFSET = 10
EOS_TOKEN = 4
PAD_TOKEN = 5

ZIP_FILE_PATH = 'known_gsgs.zip'
EXTRACT_DIR = 'cold_start_data'

# Unzip the file if it exists
if os.path.exists(ZIP_FILE_PATH):
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print(f"Extracted {ZIP_FILE_PATH} to {EXTRACT_DIR}")
else:
    print(f"Warning: {ZIP_FILE_PATH} not found. Please upload it.")

def is_connected(matrix):
    binary = (matrix > 0).astype(int)
    G = nx.from_numpy_array(binary)
    return nx.is_connected(G)

def matrix_to_sequence(matrix, num_vertices=7):
    seq = [num_vertices + VERTEX_OFFSET]
    for i in range(1, num_vertices):
        row_connections = matrix[i, :i]
        seq.extend(row_connections.tolist())
    seq.append(EOS_TOKEN)
    return seq

def matrix_to_upper_triangle(matrix):
    row_indices, col_indices = np.triu_indices(matrix.shape[0], k=1)
    return matrix[row_indices, col_indices].tolist()

# Load valid random graphs from cold start
print("Loading valid graphs of ALL sizes from cold start data...")
sequences = []

# Recursively search for all .txt files
txt_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.txt"), recursive=True)
print(f"Found {len(txt_files)} text files.")

for file in txt_files:
    if 'readme' in file.lower() or 'leaves' in file.lower():
        continue
    try:
        matrix = np.loadtxt(file, dtype=int)
        if len(matrix.shape) == 2 and matrix.shape[0] == matrix.shape[1] and is_connected(matrix):
            v = matrix.shape[0]
            seq = matrix_to_sequence(matrix, num_vertices=v)
            sequences.append(torch.tensor(seq, dtype=torch.long))
    except Exception as e:
        pass

print(f"Done! Loaded {len(sequences)} valid graphs across all sizes.")

if len(sequences) > 0:
    # Determine sequence length dynamically based on the largest graph loaded
    max_loaded_len = max(len(seq) for seq in sequences)
    # Make sure it's at least large enough for NUM_VERTICES target
    min_required_len = 1 + (NUM_VERTICES * (NUM_VERTICES - 1) // 2) + 1
    SEQ_LENGTH = max(max_loaded_len, min_required_len)

    padded = [F.pad(seq, (0, SEQ_LENGTH - len(seq)), value=PAD_TOKEN) for seq in sequences]
    dataset_tensor = torch.stack(padded)
    print(f"Dataset shape: {dataset_tensor.shape}")
    print(f"Using SEQ_LENGTH = {SEQ_LENGTH}")
else:
    print("Dataset is empty! Check your known_gsgs.zip file.")
    NUM_EDGES_UPPER = NUM_VERTICES * (NUM_VERTICES - 1) // 2
    SEQ_LENGTH = 1 + NUM_EDGES_UPPER + 1
    dataset_tensor = torch.empty((0, SEQ_LENGTH), dtype=torch.long)


Extracted known_gsgs.zip to cold_start_data
Loading valid graphs of ALL sizes from cold start data...
Found 2022 text files.
Done! Loaded 2021 valid graphs across all sizes.
Dataset shape: torch.Size([2021, 155])
Using SEQ_LENGTH = 155


In [8]:
import torch
import torch.nn as nn
import math

# Define the GraphGPT model
class GraphGPT(nn.Module):
    def __init__(self, vocab_size=6, max_seq_len=200, d_model=128, nhead=4, num_layers=4, pad_token=5):
        super(GraphGPT, self).__init__()
        self.pad_token = pad_token
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(d_model, vocab_size)

    def generate_causal_mask(self, sz):
        return torch.triu(torch.ones(sz, sz), diagonal=1).bool()

    def forward(self, x):
        batch_size, seq_len = x.size()
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x_emb = self.token_embedding(x) + self.position_embedding(positions)
        x_emb = x_emb * math.sqrt(self.d_model)
        causal_mask = self.generate_causal_mask(seq_len).to(x.device)
        padding_mask = (x == self.pad_token)
        transformer_out = self.transformer(x_emb, mask=causal_mask, src_key_padding_mask=padding_mask)
        return self.output_layer(transformer_out)


In [9]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast

vocab_size = 30
max_length = SEQ_LENGTH  # Dynamically adapts to the dataset tensor
pad_token = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_size = int(0.9 * len(dataset_tensor))
train_data, val_data = torch.utils.data.random_split(dataset_tensor, [train_size, len(dataset_tensor) - train_size])
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

model = GraphGPT(vocab_size=vocab_size, max_seq_len=max_length, pad_token=pad_token).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=pad_token)
scaler = GradScaler('cuda')

epochs = 100
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        x, y = batch[:, :-1], batch[:, 1:]
        optimizer.zero_grad()
        with autocast('cuda'):
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            x_v, y_v = batch[:, :-1], batch[:, 1:]
            logits_v = model(x_v)
            loss_v = criterion(logits_v.reshape(-1, vocab_size), y_v.reshape(-1))
            total_val_loss += loss_v.item()
    avg_val_loss = total_val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1:03d} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_graph_gpt.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break


Epoch 001 | Train: 1.2071 | Val: 0.5773 | LR: 0.000100
Epoch 002 | Train: 0.5041 | Val: 0.3952 | LR: 0.000100
Epoch 003 | Train: 0.3726 | Val: 0.3218 | LR: 0.000100
Epoch 004 | Train: 0.3246 | Val: 0.2875 | LR: 0.000100
Epoch 005 | Train: 0.2952 | Val: 0.2642 | LR: 0.000100
Epoch 006 | Train: 0.2756 | Val: 0.2469 | LR: 0.000100
Epoch 007 | Train: 0.2595 | Val: 0.2323 | LR: 0.000100
Epoch 008 | Train: 0.2488 | Val: 0.2229 | LR: 0.000100
Epoch 009 | Train: 0.2388 | Val: 0.2152 | LR: 0.000100
Epoch 010 | Train: 0.2304 | Val: 0.2073 | LR: 0.000100
Epoch 011 | Train: 0.2239 | Val: 0.2019 | LR: 0.000100
Epoch 012 | Train: 0.2180 | Val: 0.1975 | LR: 0.000100
Epoch 013 | Train: 0.2129 | Val: 0.1933 | LR: 0.000100
Epoch 014 | Train: 0.2084 | Val: 0.1894 | LR: 0.000100
Epoch 015 | Train: 0.2036 | Val: 0.1844 | LR: 0.000100
Epoch 016 | Train: 0.1988 | Val: 0.1811 | LR: 0.000100
Epoch 017 | Train: 0.1964 | Val: 0.1776 | LR: 0.000100
Epoch 018 | Train: 0.1919 | Val: 0.1745 | LR: 0.000100
Epoch 019 

In [10]:
def generate_graph_sequences_batch(model, target_num_nodes, batch_size=1, max_len=23, eos_token=4, pad_token=5):
    model.eval()
    device = next(model.parameters()).device
    start_token = target_num_nodes + VERTEX_OFFSET
    seqs = torch.full((batch_size, 1), start_token, dtype=torch.long, device=device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    with torch.no_grad():
        for _ in range(max_len - 1):
            logits = model(seqs)
            next_token_logits = logits[:, -1, :]

            # Mask invalid tokens
            next_token_logits[:, pad_token:VERTEX_OFFSET] = -float('inf')
            next_token_logits[:, VERTEX_OFFSET:] = -float('inf')

            probs = F.softmax(next_token_logits, dim=-1)
            next_tokens = torch.multinomial(probs, num_samples=1)

            seqs = torch.cat([seqs, next_tokens], dim=1)

            is_eos = (next_tokens.squeeze(-1) == eos_token)
            finished = finished | is_eos

            if finished.all():
                break

    result_seqs = []
    for i in range(batch_size):
        row = seqs[i].tolist()
        if eos_token in row:
            row = row[:row.index(eos_token) + 1]
        result_seqs.append(row)
    return result_seqs


def decode_to_matrix(seq, eos_token=4, pad_token=5):
    clean_seq = [t for t in seq[1:] if t not in (eos_token, pad_token)]
    L = len(clean_seq)
    if L == 0:
        return np.zeros((1, 1), dtype=np.int32)
    N = int(np.floor((1 + np.sqrt(1 + 8 * L)) / 2))
    matrix = np.zeros((N, N), dtype=np.int32)
    idx = 0
    for i in range(1, N):
        for j in range(i):
            if idx < L:
                matrix[i, j] = clean_seq[idx]
                matrix[j, i] = clean_seq[idx]
                idx += 1
    return matrix


# Quick validation check with batching
num_graphs = 100
target_nodes = 9
valid_graphs = 0

gen_seqs = generate_graph_sequences_batch(model, target_num_nodes=target_nodes, batch_size=num_graphs)
for gen_seq in gen_seqs:
    adj_matrix = decode_to_matrix(gen_seq)
    G = nx.from_numpy_array((adj_matrix > 0).astype(int))
    valid_edges = np.all((adj_matrix >= 0) & (adj_matrix <= 2))
    if G.number_of_nodes() > 0 and nx.is_connected(G) and valid_edges:
        valid_graphs += 1
print(f"Validity check: {valid_graphs}/{num_graphs} generated graphs are valid.")


Validity check: 86/100 generated graphs are valid.


Julia setup below. (takes a bit)

In [18]:

import subprocess, sys
print("Installing Julia + juliacall...")
subprocess.run([sys.executable, "-m", "pip", "install", "juliacall", "-q"], check=True)

from juliacall import Main as jl
jl.seval("import Pkg")
for pkg in ["ChipFiring", "Graphs", "TreeWidthSolver"]:
    print(f"  Setting up {pkg}...")
    jl.seval(f'if !haskey(Pkg.project().dependencies, "{pkg}") Pkg.add("{pkg}") end')
jl.seval("using ChipFiring; using Graphs; using TreeWidthSolver")
jl.seval('function fast_convert(np_array) return convert(Matrix{Int64}, np_array) end')
fast_convert = jl.fast_convert
print("Julia setup complete!")

N_REWARD = NUM_VERTICES
MYN = N_REWARD * (N_REWARD - 1) // 2
MAX_EDGES_TO_CHECK = 18
TRIU_I, TRIU_J = np.triu_indices(N_REWARD, k=1)
score_cache = {}


def compute_gon_2_subdivision(g, gonality, n, num_nonzero_edges):
    if check_non_gsg(g, gonality, n, num_nonzero_edges):
        return gonality
    sub_g = jl.subdivide(g, 2)
    rank_to_check = gonality
    while rank_to_check > 0:
        if jl.compute_gonality(sub_g, min_d=rank_to_check, max_d=rank_to_check) == -1:
            return rank_to_check + 1
        rank_to_check -= 1
    return jl.compute_gonality(sub_g)


def check_non_gsg(g, gonality, n, num_edges):
    if n <= 5:
        return True
    if gonality <= 4:
        return True
    if num_edges > MAX_EDGES_TO_CHECK:
        return True
    if jl.compute_genus(g) <= 5:
        return True
    return False


def is_connected_fast(edges, n):
    parent = list(range(n))

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj

    for idx in range(len(edges)):
        if edges[idx] > 0:
            union(TRIU_I[idx], TRIU_J[idx])
    return all(find(i) == find(0) for i in range(1, n))


def calcScore(state):
    edges = state[:MYN]
    state_key = tuple(edges)

    if state_key in score_cache:
        return score_cache[state_key]

    num_edges_total_weight = np.sum(edges)
    if num_edges_total_weight == 0 or not is_connected_fast(edges, N_REWARD):
        result = (0.0, False, {"reason": "disconnected_or_empty"})
        score_cache[state_key] = result
        return result

    adj = np.zeros((N_REWARD, N_REWARD), dtype=int)
    adj[TRIU_I, TRIU_J] = edges
    adj[TRIU_J, TRIU_I] = edges

    try:
        jl_matrix = fast_convert(adj)
        graph = jl.ChipFiringGraph(jl_matrix)
        gon = int(jl.compute_gonality(graph))

        if gon < 5:
            result = (
                float(gon),
                False,
                {"gon": gon, "gon2": None, "treewidth": None, "reason": "gon_lt_5"},
            )
            score_cache[state_key] = result
            return result

        gon2 = compute_gon_2_subdivision(graph, gon, N_REWARD, num_edges_total_weight)
        genus = int(jl.compute_genus(graph))
        treewidth = int(jl.exact_treewidth(jl.SimpleGraph(graph.adj_matrix)))
        is_gsg = gon != gon2

        reward = (
            200 * (gon - gon2)
            + 10 * (gon - treewidth)
            + 10 * (gon - np.floor((genus + 3)/2))
        )

        info = {
            "gon": gon,
            "gon2": gon2,
            "treewidth": treewidth,
            "num_edges": int(num_edges_total_weight),
        }

        result = (reward, is_gsg, info)
        score_cache[state_key] = result
        return result

    except Exception as e:
        result = (0.0, False, {"reason": "error", "error": str(e)})
        score_cache[state_key] = result
        return result


def sequence_to_upper_triangle(seq):
    matrix = decode_to_matrix(seq)
    if matrix.shape[0] != N_REWARD:
        return None, None
    if not np.all((matrix >= 0) & (matrix <= 2)):
        return None, None
    return matrix[TRIU_I, TRIU_J].tolist(), matrix


print("Reward function ready!")

Installing Julia + juliacall...
  Setting up ChipFiring...
  Setting up Graphs...
  Setting up TreeWidthSolver...
Julia setup complete!
Reward function ready!


In [ ]:
##############################################################
# 10-TRIAL BENCHMARK
# Run Cells 1-6 first.
##############################################################

import time
import math
import random
import uuid
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from networkx.algorithms.graph_hashing import weisfeiler_lehman_graph_hash


def retrain_model(model, data, epochs=20, lr=1e-4):
    dev = next(model.parameters()).device
    ts = max(1, int(0.9 * len(data))) if len(data) > 1 else len(data)
    td, vd = torch.utils.data.random_split(data, [ts, len(data) - ts])
    tl = DataLoader(td, batch_size=32, shuffle=True)
    vl = DataLoader(vd, batch_size=32, shuffle=False)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    crit = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
    best = float("inf")
    pat = 0

    for ep in range(epochs):
        model.train()
        for b in tl:
            b = b.to(dev)
            x, y = b[:, :-1], b[:, 1:]
            opt.zero_grad()
            loss = crit(model(x).reshape(-1, 30), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        vloss = 0
        with torch.no_grad():
            for b in vl:
                b = b.to(dev)
                xv, yv = b[:, :-1], b[:, 1:]
                vloss += crit(model(xv).reshape(-1, 30), yv.reshape(-1)).item()

        if len(vl) > 0:
            av = vloss / len(vl)
            if av < best:
                best = av
                pat = 0
            else:
                pat += 1
                if pat >= 3:
                    break
        else:
            # If no validation set, just train for all epochs
            pass
    return model


def graph_hash(matrix):
    return weisfeiler_lehman_graph_hash(nx.from_numpy_array(matrix), edge_attr='weight')


print("Computing hashes of known graphs to ensure uniqueness...")
known_gsg_hashes = set()
for seq in sequences:
    mx = decode_to_matrix(seq.tolist())
    if mx.shape[0] == NUM_VERTICES:
        known_gsg_hashes.add(graph_hash(mx))
print(f"Found {len(known_gsg_hashes)} known unique hashes for V={NUM_VERTICES}.")


NUM_TRIALS = 10
MAX_ROUNDS = 10000  # Significantly increased so the 1200s time limit acts as the true bound
GRAPHS_PER_BATCH = 800
TOP_K_PERCENT = 0.05
RETRAIN_EPOCHS = 20
TIME_LIMIT = 1200

trial_times = []
trial_timing = []
trial_gsg_counts = []

print("=" * 60)
print(f"  RUNNING {NUM_TRIALS} TRIALS | V={NUM_VERTICES}")
print(f"  Pool approach: score all, retrain on top {TOP_K_PERCENT*100:.0f}%")
print(f"  Max time per trial: {TIME_LIMIT}s")
print("=" * 60)

for trial in range(1, NUM_TRIALS + 1):
    print(f"\n{'='*60}")
    print(f"  TRIAL {trial}/{NUM_TRIALS}")
    print(f"{'='*60}")

    trial_start = time.time()
    score_cache.clear()
    timing = {
        "data": 0.0,
        "initial_training": 0.0,
        "generation": 0.0,
        "scoring": 0.0,
        "retraining": 0.0,
    }
    gsgs_found_this_trial = 0
    saved_hashes = set(known_gsg_hashes) # Initialize with known graphs

    # Use loaded cold start training data
    t_data = time.time()
    trial_seqs = random.sample(sequences, min(3000, len(sequences))) if len(sequences) > 0 else []
    ds = torch.stack([F.pad(s, (0, SEQ_LENGTH - len(s)), value=PAD_TOKEN) for s in trial_seqs]) if trial_seqs else torch.empty((0, SEQ_LENGTH))
    timing["data"] += time.time() - t_data
    print(f"  Data: {len(ds)} graphs ({timing['data']:.1f}s)")

    # Score all initial cold start graphs to build the scored pool
    t_score_init = time.time()
    scored_pool = []
    for i, s in enumerate(trial_seqs):
        seq_list = s.tolist()
        ut, mx = sequence_to_upper_triangle(seq_list)
        if ut is not None:
            rw, is_gsg, info = calcScore(ut)
            st = F.pad(s, (0, SEQ_LENGTH - len(s)), value=PAD_TOKEN)
            scored_pool.append((rw, is_gsg, info, st))
    timing["scoring"] += time.time() - t_score_init
    print(f"  Scored initial pool: {len(scored_pool)} graphs ({time.time() - t_score_init:.1f}s)")

    # Fresh model + initial training on ALL of scored pool (as requested by user)
    scored_pool.sort(key=lambda x: x[0], reverse=True)
    top_k_seqs = [sp[3] for sp in scored_pool] # Use all for initial start
    train_ds = torch.stack(top_k_seqs) if top_k_seqs else torch.empty((0, SEQ_LENGTH), dtype=torch.long)

    t_train = time.time()
    mdl = GraphGPT(vocab_size=30, max_seq_len=SEQ_LENGTH, pad_token=5).to(device)
    if len(train_ds) > 0:
        opt = optim.AdamW(mdl.parameters(), lr=1e-4, weight_decay=0.01)
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, "min", patience=2, factor=0.5)
        crit = nn.CrossEntropyLoss(ignore_index=5)
        scl = torch.amp.GradScaler('cuda')

        ts = max(1, int(0.9 * len(train_ds))) if len(train_ds) > 1 else len(train_ds)
        td, vd = torch.utils.data.random_split(train_ds, [ts, len(train_ds) - ts])
        tl = DataLoader(td, batch_size=32, shuffle=True)
        vl = DataLoader(vd, batch_size=32, shuffle=False)

        best_vl = float("inf")
        no_imp = 0
        for ep in range(100):
            if len(tl) == 0: break
            mdl.train()
            for b in tl:
                b = b.to(device)
                x, y = b[:, :-1], b[:, 1:]
                opt.zero_grad()
                with torch.amp.autocast('cuda'):
                    loss = crit(mdl(x).reshape(-1, 30), y.reshape(-1))
                scl.scale(loss).backward()
                scl.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                scl.step(opt)
                scl.update()

            mdl.eval()
            vl_loss = 0
            with torch.no_grad():
                for b in vl:
                    b = b.to(device)
                    xv, yv = b[:, :-1], b[:, 1:]
                    vl_loss += crit(mdl(xv).reshape(-1, 30), yv.reshape(-1)).item()

            if len(vl) > 0:
                av = vl_loss / len(vl)
                sched.step(av)
                if av < best_vl:
                    best_vl = av
                    no_imp = 0
                else:
                    no_imp += 1
                    if no_imp >= 5:
                        break
            else:
                # If no validation set, just run all epochs without early stopping
                pass

    timing["initial_training"] += time.time() - t_train
    print(f"  Initial training done ({timing['initial_training']:.1f}s)")

    # Generate-score-retrain loop
    seen_hashes = set()

    for rnd in range(1, MAX_ROUNDS + 1):
        if time.time() - trial_start > TIME_LIMIT:
            print(f"  Time limit reached at round {rnd}")
            break

        t_gen = time.time()
        gen = []
        dups = 0
        invalid = 0

        # Batched generation
        batch_size = 200
        for i in range(0, GRAPHS_PER_BATCH, batch_size):
            curr_bs = min(batch_size, GRAPHS_PER_BATCH - i)
            batch_sqs = generate_graph_sequences_batch(mdl, target_num_nodes=NUM_VERTICES, batch_size=curr_bs, max_len=SEQ_LENGTH)
            for sq in batch_sqs:
                ut, mx = sequence_to_upper_triangle(sq)
                if ut is None:
                    invalid += 1
                    continue
                h = graph_hash(mx)
                if h in seen_hashes:
                    dups += 1
                seen_hashes.add(h)
                gen.append((sq, mx, ut))
        timing["generation"] += time.time() - t_gen

        t_score = time.time()
        for sq, mx, ut in gen:
            rw, is_gsg, info = calcScore(ut)
            st = torch.tensor(sq, dtype=torch.long)
            if len(st) < SEQ_LENGTH:
                st = F.pad(st, (0, SEQ_LENGTH - len(st)), value=PAD_TOKEN)
            elif len(st) > SEQ_LENGTH:
                st = st[:SEQ_LENGTH]
            scored_pool.append((rw, is_gsg, info, st))

            if is_gsg:
                h = graph_hash(mx)
                if h not in saved_hashes:
                    saved_hashes.add(h)
                    gsgs_found_this_trial += 1
                    elapsed = time.time() - trial_start

                    # Save the matrix to a file
                    unique_id = uuid.uuid4().hex[:8]
                    gon = info['gon']
                    gon2 = info['gon2']
                    e_count = info['num_edges']
                    filename = f"gsg_{unique_id}_gon_{gon}_sigma2_{gon2}_v_{NUM_VERTICES}_e_{e_count}.txt"
                    np.savetxt(filename, mx, fmt='%d')

                    print(f"\n  *** NEW UNIQUE GSG FOUND in round {rnd}! ***")
                    print(f"  Saved to: {filename}")
                    print(f"  Time: {elapsed:.1f}s")
                    print(f"  Reward: {rw:.2f}")
                    print(f"  gon(G): {gon}")
                    print(f"  gon(sigma_2(G)): {gon2}")
                    print(f"  treewidth: {info['treewidth']}")
                    print(f"  Upper triangle: {ut}")
                    # Continue searching, no break!

        timing["scoring"] += time.time() - t_score

        # Rank entire pool, retrain on top K%
        scored_pool.sort(key=lambda x: x[0], reverse=True)
        top_k_count = max(1, int(len(scored_pool) * TOP_K_PERCENT))
        top_k_seqs = [sp[3] for sp in scored_pool[:top_k_count]]
        best_reward = scored_pool[0][0] if scored_pool else 0

        print(
            f"  Round {rnd}: generated={len(gen)}, dups={dups}, invalid={invalid}, "
            f"pool={len(scored_pool)}, top_k={top_k_count}, best_rw={best_reward:.2f}"
        )

        t_retrain = time.time()
        if top_k_seqs:
            mdl = retrain_model(mdl, torch.stack(top_k_seqs), epochs=RETRAIN_EPOCHS)
        timing["retraining"] += time.time() - t_retrain

    t = time.time() - trial_start
    trial_timing.append(timing)
    trial_gsg_counts.append(gsgs_found_this_trial)

    print(
        "  Timing split: "
        f"data={timing['data']:.1f}s | "
        f"initial_train={timing['initial_training']:.1f}s | "
        f"generation={timing['generation']:.1f}s | "
        f"scoring={timing['scoring']:.1f}s | "
        f"retraining={timing['retraining']:.1f}s"
    )

    if gsgs_found_this_trial > 0:
        trial_times.append(t)
        print(f"  Trial {trial}: FINISHED with {gsgs_found_this_trial} UNIQUE GSGs ({t:.1f}s)")
    else:
        trial_times.append(math.inf)
        print(f"  Trial {trial}: FAILED, 0 unique GSGs found ({t:.1f}s)")


# === FINAL REPORT ===
print("\n" + "=" * 60)
print("  RESULTS SUMMARY")
print("=" * 60)

converged = [t for t in trial_times if t != math.inf]
print(f"\n  All trials:")
for i, (t, c) in enumerate(zip(trial_times, trial_gsg_counts)):
    if t != math.inf:
        print(f"    Trial {i+1}: {c} UNIQUE GSGs found in {t:.1f}s")
    else:
        print(f"    Trial {i+1}: DID NOT FIND UNIQUE GSGs ({t:.1f}s)")

total_found = sum(trial_gsg_counts)
print(f"\n  Convergence: {len(converged)}/{NUM_TRIALS} trials found at least 1 unique GSG")
print(f"  Total unique GSGs discovered across all trials: {total_found}")

if trial_timing:
    totals = {
        key: sum(t[key] for t in trial_timing)
        for key in ["data", "initial_training", "generation", "scoring", "retraining"]
    }
    measured = sum(totals.values())
    print("\n  Aggregate timing split:")
    for key, value in totals.items():
        pct = (100.0 * value / measured) if measured else 0.0
        print(f"    {key}: {value:.1f}s ({pct:.1f}%)")

print("=" * 60)


Computing hashes of known graphs to ensure uniqueness...
Found 94 known unique hashes for V=9.
  RUNNING 10 TRIALS | V=9
  Pool approach: score all, retrain on top 5%
  Max time per trial: 1200s

  TRIAL 1/10
  Data: 2021 graphs (0.0s)
  Scored initial pool: 74 graphs (16.9s)
  Initial training done (3.5s)

  *** NEW UNIQUE GSG FOUND in round 1! ***
  Saved to: gsg_114360cd_gon_6_sigma2_5_v_9_e_18.txt
  Time: 40.3s
  Reward: 230.00
  gon(G): 6
  gon(sigma_2(G)): 5
  treewidth: 3
  Upper triangle: [0, 0, 1, 0, 1, 1, 0, 2, 0, 0, 1, 0, 2, 0, 0, 1, 0, 2, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 2]

  *** NEW UNIQUE GSG FOUND in round 1! ***
  Saved to: gsg_583b4194_gon_6_sigma2_5_v_9_e_17.txt
  Time: 45.6s
  Reward: 230.00
  gon(G): 6
  gon(sigma_2(G)): 5
  treewidth: 3
  Upper triangle: [0, 0, 1, 0, 1, 2, 0, 2, 0, 0, 1, 0, 1, 0, 0, 0, 0, 2, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 2]
  Round 1: generated=732, dups=87, invalid=68, pool=806, top_k=40, best_rw=230.00

  *

### Save all found GSGs to a zip file

In [13]:
import zipfile
import glob
import os

output_zip_name = f"found_gsgs_V{NUM_VERTICES}.zip"

# Get a list of all gsg_*.txt files in the current directory
gsg_files = glob.glob("gsg_*.txt")

if gsg_files:
    with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in gsg_files:
            zipf.write(file, os.path.basename(file))
    print(f"Successfully created '{output_zip_name}' containing {len(gsg_files)} GSG files.")
else:
    print("No GSG files found to zip.")


Successfully created 'found_gsgs_V9.zip' containing 18 GSG files.
